# Set-Up

In [ ]:
# Imports for Generating & Viewing Data #

import h5py
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

In [ ]:
# Constants #
DATA_DIR = "../Data/"

GAUSSIAN_X = 48
GAUSSIAN_Y = 48

TRAINING_GROWTHS = 7
VALIDATION_GROWTHS = 2
TEST_GROWTHS = 1
IMAGES_PER_GROWTH = 10000

# params: [center_x, center_y, std_x, std_y, sin2theta, cos2theta, intensity]
scaling_arr = np.array([48.0, 48.0, 10.0, 10.0, 1.0, 1.0, 1.0], dtype=np.float32)

SEED = 0
rng = np.random.default_rng(SEED)

In [ ]:
# Read H5 File #
RHEED_data_file = DATA_DIR + "RHEED_4848_test6.h5"
SPOTS = ["spot_1", "spot_2", "spot_3"]
# SPOTS = ['spot_2']

h5 = h5py.File(RHEED_data_file, "r")
growths = list(h5.keys())
rng.shuffle(growths)  # Shuffle Growths
training_growths = growths[:TRAINING_GROWTHS]
validation_growths = growths[TRAINING_GROWTHS : TRAINING_GROWTHS + VALIDATION_GROWTHS]
test_growths = growths[
    TRAINING_GROWTHS + VALIDATION_GROWTHS : TRAINING_GROWTHS
    + VALIDATION_GROWTHS
    + TEST_GROWTHS
]

print("Raw Train Data Set:")
raw_train_data_dict = {}
for growth in training_growths:
    for spot in SPOTS:
        indices = rng.choice(
            h5[growth][spot].shape[0], size=IMAGES_PER_GROWTH, replace=False
        )
        indices.sort()
        raw_train_data_dict[(growth, spot)] = np.expand_dims(
            h5[growth][spot][indices], -1
        ).astype(np.float32)
        print(
            f"[Growth]: {growth:<12}, [Spot]: {spot}, [Shape]: {raw_train_data_dict[(growth, spot)].shape}"
        )

print("Raw Validation Data Set:")
raw_validation_data_dict = {}
for growth in validation_growths:
    for spot in SPOTS:
        indices = rng.choice(
            h5[growth][spot].shape[0], size=IMAGES_PER_GROWTH, replace=False
        )
        indices.sort()
        raw_validation_data_dict[(growth, spot)] = np.expand_dims(
            h5[growth][spot][indices], -1
        ).astype(np.float32)
        print(
            f"[Growth]: {growth:<12}, [Spot]: {spot}, [Shape]: {raw_validation_data_dict[(growth, spot)].shape}"
        )

print("Raw Test Data Set:")
raw_test_data_dict = {}
for growth in test_growths:
    for spot in SPOTS:
        indices = rng.choice(
            h5[growth][spot].shape[0], size=IMAGES_PER_GROWTH, replace=False
        )
        indices.sort()
        raw_test_data_dict[(growth, spot)] = np.expand_dims(
            h5[growth][spot][indices], -1
        ).astype(np.float32)
        print(
            f"[Growth]: {growth:<12}, [Spot]: {spot}, [Shape]: {raw_test_data_dict[(growth, spot)].shape}"
        )

In [ ]:
# Aggregate Data #

raw_train_data = np.concatenate(list(raw_train_data_dict.values()))
raw_validation_data = np.concatenate(list(raw_validation_data_dict.values()))
raw_test_data = np.concatenate(list(raw_test_data_dict.values()))

print(f"[Train Data Set Shape]:      {raw_train_data.shape}")
print(f"[Validation Data Set Shape]: {raw_validation_data.shape}")
print(f"[Test Data Set Shape]:       {raw_test_data.shape}")

In [ ]:
# Image Normalization #


def gen_data(num_images: int, data: np.ndarray) -> np.ndarray:
    idx = rng.integers(0, data.shape[0], size=num_images)
    imgs = data[idx].astype(np.float32)
    bg = imgs.reshape(num_images, -1).min(axis=1).reshape(num_images, 1, 1, 1)
    imgs = np.clip(imgs - bg, 0.0, None) / 256.0
    return imgs

# Train / Load Qkeras Model

In [ ]:
# Imports for Training #

import tensorflow as tf
import keras.backend as K
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPool2D,
    Flatten,
    Dense,
    Concatenate,
    BatchNormalization,
    Add,
    AveragePooling2D,
    Activation,
)
from tensorflow.keras.activations import relu, sigmoid
from tensorflow.keras.models import Model
from qkeras import QConv2D, QActivation, QDense
from qkeras.quantizers import (
    quantized_bits,
    quantized_relu,
    quantized_sigmoid,
    quantized_tanh,
)

In [ ]:
# Create TF Datasets #
# TODO: look into using tf.data.Dataset.from_generator
TRAINING_DATASET_SIZE = 100000
VALIDATION_DATASET_SIZE = 10000
TEST_DATASET_SIZE = 10000
BATCH_SIZE = 128

train_img_arr = gen_data(TRAINING_DATASET_SIZE, raw_train_data)
val_img_arr = gen_data(VALIDATION_DATASET_SIZE, raw_validation_data)
test_img_arr = gen_data(TEST_DATASET_SIZE, raw_test_data)

print(f"[Train Images Shape]:      {train_img_arr.shape}")
print(f"[Validation Images Shape]: {val_img_arr.shape}")
print(f"[Test Images Shape]:       {test_img_arr.shape}")


def is_valid_image(img):  # Filter out Bad Crops
    img_2d = img[:, :, 0]

    # Condition 1: peak must be bright enough to be a real spot
    if img_2d.max() < 0.1:
        return False

    return True


# Apply to training data
valid_mask = np.array(
    [is_valid_image(train_img_arr[i]) for i in range(len(train_img_arr))]
)
print(f"Valid: {valid_mask.sum()} / {len(valid_mask)} ({100 * valid_mask.mean():.1f}%)")
train_img_arr = train_img_arr[valid_mask]

valid_mask = np.array([is_valid_image(val_img_arr[i]) for i in range(len(val_img_arr))])
print(f"Valid: {valid_mask.sum()} / {len(valid_mask)} ({100 * valid_mask.mean():.1f}%)")
val_img_arr = val_img_arr[valid_mask]

valid_mask = np.array(
    [is_valid_image(test_img_arr[i]) for i in range(len(test_img_arr))]
)
print(f"Valid: {valid_mask.sum()} / {len(valid_mask)} ({100 * valid_mask.mean():.1f}%)")
test_img_arr = test_img_arr[valid_mask]

train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_img_arr, train_img_arr))
    .shuffle(TRAINING_DATASET_SIZE, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((val_img_arr, val_img_arr))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# Qkeras Model Architecture #
# TODO: Quantize Dense Layers?
TOTAL_BITS = 8
INTEGER_BITS = 0

w_quant = quantized_bits(
    TOTAL_BITS, INTEGER_BITS, symmetric=False, keep_negative=True, alpha=1
)
relu_quant = quantized_relu(TOTAL_BITS, INTEGER_BITS)
sigmoid_quant = quantized_sigmoid(TOTAL_BITS)
tanh_quant = quantized_tanh(TOTAL_BITS)

input_layer = Input(shape=(GAUSSIAN_Y, GAUSSIAN_X, 1), name="InputLayer")

x = QConv2D(
    filters=6,
    kernel_size=3,
    strides=1,
    padding="valid",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(input_layer)
x = BatchNormalization(axis=-1, momentum=0.99, epsilon=1e-03)(x)
x = QActivation(relu_quant)(x)
x = MaxPool2D(pool_size=2, strides=2)(x)

x = QConv2D(
    filters=8,
    kernel_size=3,
    strides=1,
    padding="valid",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
x = BatchNormalization(axis=-1, momentum=0.99, epsilon=1e-03)(x)
x = QActivation(relu_quant)(x)
x = MaxPool2D(pool_size=2, strides=2)(x)

x = QConv2D(
    filters=8,
    kernel_size=3,
    strides=1,
    padding="valid",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
x = BatchNormalization(axis=-1, momentum=0.99, epsilon=1e-03)(x)
x = QActivation(relu_quant)(x)
x = MaxPool2D(pool_size=3, strides=3)(x)

x = Flatten(name="flatten")(x)  # 4*4*32 = 512
x = Dense(units=28, activation="relu")(x)

# Split head to constrain outputs:
x_center = Dense(2)(x)
x_center = Activation("sigmoid")(x_center)

x_std = Dense(2)(x)
x_std = Activation("softplus")(x_std)

x_theta = Dense(2)(x)
x_theta = Activation("tanh")(x_theta)

x_intensity = Dense(1)(x)
x_intensity = Activation("sigmoid")(x_intensity)

x_p1 = Concatenate(axis=-1, name="params_2")([x_center, x_std])
x_p2 = Concatenate(axis=-1, name="params_3")([x_theta, x_intensity])
x_params = Concatenate(axis=-1, name="params")([x_p1, x_p2])  # 7-dim

# Don't split head to constrain outputs:
# x_params    = Dense(units=7, activation='sigmoid', name='out')(x)  # 7-dim

gaussian_qk = Model(inputs=input_layer, outputs=x_params, name="gaussian")

In [ ]:
# Tensorflow Functions #
BG_WEIGHT = 0.13


def recon_loss(y_true, y_pred):
    W = y_true + BG_WEIGHT
    return tf.reduce_mean(tf.reduce_sum(W * tf.square(y_true - y_pred), axis=[1, 2, 3]))


scaling_tf = tf.constant(scaling_arr, dtype=tf.float32)
X_mesh_tf, Y_mesh_tf = tf.meshgrid(
    tf.range(GAUSSIAN_X, dtype=tf.float32),
    tf.range(GAUSSIAN_Y, dtype=tf.float32),
)
X_mesh_tf = X_mesh_tf[tf.newaxis, :, :, tf.newaxis]
Y_mesh_tf = Y_mesh_tf[tf.newaxis, :, :, tf.newaxis]


def gaussian_gen_tf(params_norm):
    # params_norm ordering: [cx, cy, sx, sy, sin2t, cos2t, intensity]
    params = params_norm * scaling_tf
    cx = tf.reshape(params[:, 0], (-1, 1, 1, 1))
    cy = tf.reshape(params[:, 1], (-1, 1, 1, 1))
    sx = tf.reshape(params[:, 2], (-1, 1, 1, 1))
    sy = tf.reshape(params[:, 3], (-1, 1, 1, 1))
    s2t = tf.reshape(params[:, 4], (-1, 1, 1, 1))
    c2t = tf.reshape(params[:, 5], (-1, 1, 1, 1))
    ity = tf.reshape(params[:, 6], (-1, 1, 1, 1))

    norm = tf.sqrt(tf.square(s2t) + tf.square(c2t) + 1e-6)
    s2t = s2t / norm
    c2t = c2t / norm

    cos_sqr = 0.5 * (1.0 + c2t)
    sin_sqr = 0.5 * (1.0 - c2t)
    sin_cos = 0.5 * s2t

    std_x_sqrd = tf.square(sx) + 1e-6
    std_y_sqrd = tf.square(sy) + 1e-6

    a = (cos_sqr) / (2.0 * std_x_sqrd) + (sin_sqr) / (2.0 * std_y_sqrd)
    b = -1.0 * (sin_cos) / (2.0 * std_x_sqrd) + (sin_cos) / (2.0 * std_y_sqrd)
    c = (sin_sqr) / (2.0 * std_x_sqrd) + (cos_sqr) / (2.0 * std_y_sqrd)

    dx = X_mesh_tf - cx
    dy = Y_mesh_tf - cy
    return ity * tf.exp(-(a * tf.square(dx) + 2.0 * b * dx * dy + c * tf.square(dy)))


x_reconstruction = tf.keras.layers.Lambda(gaussian_gen_tf, name="reconstruction")(
    gaussian_qk.output
)
training_model = Model(
    inputs=gaussian_qk.input,
    outputs=x_reconstruction,
    name="gaussian_training",
)

In [ ]:
# TODO: Fix Load
TRAIN_MODEL = True
LOAD_MODEL = False

MODEL_NAME = ""

NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 1e-4


if TRAIN_MODEL and LOAD_MODEL:
    print("Are you sure you want to Train & Load ?")

elif TRAIN_MODEL:
    adam_optimizer = tf.keras.optimizers.Adam(
        learning_rate=1e-3,
        global_clipnorm=1.0,
    )

    training_model.compile(
        optimizer=adam_optimizer,
        loss=recon_loss,
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=MIN_DELTA,
        restore_best_weights=True,
        verbose=1,
    )

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1,
    )

    history = training_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=NUM_EPOCHS,
        callbacks=[early_stop, reduce_lr],
        verbose=1,
    )

elif LOAD_MODEL:
    pass

gaussian_qk.summary()

In [ ]:
# Save Model #
SAVE_MODEL = False

if SAVE_MODEL:
    gaussian_qk.save(f"Models/{MODEL_NAME}")

# Test QKeras Model

In [ ]:
# Util Functions #

X, Y = np.meshgrid(np.arange(GAUSSIAN_X), np.arange(GAUSSIAN_Y))


def gaussian_gen(
    center_x: float, center_y: float, std_x: float, std_y: float, theta: float, X=X, Y=Y
) -> np.ndarray:

    cos_theta_sqrd = np.power(np.cos(theta), 2)
    sin_theta_sqrd = np.power(np.sin(theta), 2)
    sin_cos_theta = np.sin(theta) * np.cos(theta)

    std_x_sqrd = np.power(std_x, 2)
    std_y_sqrd = np.power(std_y, 2)

    a = (cos_theta_sqrd) / (2 * std_x_sqrd) + (sin_theta_sqrd) / (2 * std_y_sqrd)
    b = -1 * (sin_cos_theta) / (2 * std_x_sqrd) + (sin_cos_theta) / (2 * std_y_sqrd)
    c = (sin_theta_sqrd) / (2 * std_x_sqrd) + (cos_theta_sqrd) / (2 * std_y_sqrd)

    gaussian = np.exp(
        -(
            a * (X - center_x) ** 2
            + 2 * b * (X - center_x) * (Y - center_y)
            + c * (Y - center_y) ** 2
        )
    )

    return np.expand_dims(gaussian, -1)

In [ ]:
# Predict on Test Set #

predictions_qk = gaussian_qk.predict(test_img_arr)

print(f"[Prediction Shape]: {test_img_arr.shape}")

# Convert to HLS

In [ ]:
# Imports for HLS #

import hls4ml
from hls4ml.utils import config_from_keras_model
from pprint import pprint

import os

xilinx_vitis = "/home/tools/Xilinx/2025/2025.2/Vitis"
xilinx_viv = "/home/tools/Xilinx/2025/2025.2/Vivado"
xilinx_hls = "/home/tools/Xilinx/2025/2025.2/Vitis"

# prepend Vitis bin directory to PATH
# os.environ["PATH"] = f"{xilinx_vitis}/bin:" + os.environ["PATH"]

os.environ["XILINX_HLS"] = xilinx_hls
os.environ["XILINX_VITIS"] = xilinx_vitis
os.environ["XILINX_VIVADO"] = xilinx_viv

In [ ]:
# Utils for HLS #

data = test_img_arr[:3].reshape(3, -1)
np.savetxt("Utils/Real/tb_input_features.dat", data, fmt="%.6f")

data = predictions_qk[:3].reshape(3, -1)
np.savetxt("Utils/Real/tb_output_predictions.dat", data, fmt="%.6f")

In [ ]:
# HLS4ML Config #
# TODO: look into pipeline
# TODO: verify config quantization

config = config_from_keras_model(gaussian_qk, granularity="name", backend="Vitis")

# Fifo Depth Optimization (greatly reduces BRAM)
config["Flows"] = ["vitis:fifo_depth_optimization"]
hls4ml.model.optimizer.get_optimizer("vitis:fifo_depth_optimization").configure(
    profiling_fifo_depth=1000
)

# Strategy
config["Model"]["Strategy"] = "Latency"
# config['Model']['PipelineStyle'] = 'pipeline'

# Reuse Factor
# config["Model"]["ReuseFactor"] = 4
# for layer_cfg in config["LayerName"].values():
#     if "ReuseFactor" in layer_cfg:
#         layer_cfg["ReuseFactor"] = 4

pprint(config)

In [ ]:
# Compile #
# Make sure you copy over the TB Data found in Utils for FOLO Depth Opt.

hls_model = hls4ml.converters.convert_from_keras_model(
    gaussian_qk,
    hls_config=config,
    output_dir="gaussian_hls4ml_Q_Real",
    io_type="io_stream",  # io_parallel
    clock_period=2.0,
    clock_uncertainty="12.5%",
    backend="Vitis",
    part="xcku035-fbva676-2-e",
    project_name="gaussian",
)

hls_model.compile()

In [ ]:
# Predict #

predictions_hls4ml = hls_model.predict(test_img_arr).reshape(test_img_arr.shape[0], 7)

print(f"[Prediction Shape]: {predictions_hls4ml.shape}")

In [ ]:
# Visually Compare Predictions #
index = np.random.randint(low=0, high=test_img_arr.shape[0])

tf_real = predictions_qk[index] * scaling_arr
hls_real = predictions_hls4ml[index] * scaling_arr

# Recover theta from (sin2t, cos2t) via arctan2.
tf_theta = 0.5 * np.arctan2(tf_real[4], tf_real[5])
hls_theta = 0.5 * np.arctan2(hls_real[4], hls_real[5])

# Reconstruct image using parameters
tf_recon = gaussian_gen(tf_real[0], tf_real[1], tf_real[2], tf_real[3], tf_theta)
hls_recon = gaussian_gen(hls_real[0], hls_real[1], hls_real[2], hls_real[3], hls_theta)

fig, axes = plt.subplots(1, 3)
im0 = axes[0].imshow(test_img_arr[index], cmap="viridis", interpolation="none")
axes[0].set_title("Test Label")
axes[0].axis("off")

im1 = axes[1].imshow(tf_recon, cmap="viridis", interpolation="none")
axes[1].set_title("QK Prediction")
axes[1].axis("off")

im2 = axes[2].imshow(hls_recon, cmap="viridis", interpolation="none")
axes[2].set_title("HLS Prediction")
axes[2].axis("off")

plt.show()